# Improve Your LLM Judge with Human Feedback

Close the loop between automated evals and human judgment: run a custom eval, annotate its mistakes, refine the criteria, and re-evaluate to measure improvement.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/future-agi/cookbooks/blob/cookbook/quickstart-notebooks/use-cases/feedback-loop-eval.ipynb)

| Time | Difficulty |
|------|------------|
| 30 min | Intermediate |

You have a custom eval that scores your LLM's output automatically, but it disagrees with human judgment too often. Sarcasm gets flagged as harmful. Slang gets misread. The eval is useful, but it has blind spots.

The fix is not to replace the eval. It is to teach it. This cookbook walks you through one complete feedback cycle: run the eval, have humans annotate the mistakes, extract the patterns from those corrections, update the criteria, and re-run to confirm the improvement.

**Prerequisites**
- FutureAGI account: [app.futureagi.com](https://app.futureagi.com)
- API keys: `FI_API_KEY` and `FI_SECRET_KEY` (see [Get your API keys](https://docs.futureagi.com/docs/admin-settings))
- Python 3.9+

## Install

In [ ]:
!pip install futureagi ai-evaluation pandas

In [ ]:
import os

os.environ["FI_API_KEY"] = "your-api-key"
os.environ["FI_SECRET_KEY"] = "your-secret-key"

## Step 1: Define a simple baseline eval (intentionally naive)

Start with a custom eval that has a naive Rule Prompt. In this example, you are building a content moderation eval for user-generated posts. The first version is intentionally basic so you can see where it fails and improve it.

**Create the eval in the dashboard:**

1. Go to [app.futureagi.com](https://app.futureagi.com) → **Evals** (left sidebar under BUILD)
2. Click the **Create your own evals** card
3. **Step 1 (Add Details):** Enter name `content_appropriateness`, select template type **Use Future AGI Agents**, then select language model `turing_small`
4. **Step 2 (Rule Prompt):** Paste the following. Variables wrapped in `{{}}` are automatically extracted.

```
You are a content moderator for a social media platform.

Evaluate this user post: {{post_content}}

Mark PASS if the post is appropriate for a general audience.
Mark FAIL if the post contains:
- Hate speech or slurs targeting any group
- Explicit threats of violence
- Graphic sexual content
- Harassment directed at a specific individual

Return a clear PASS/FAIL decision with a short reason.
```

5. **Step 3 (Output Type):** Select **Pass/Fail**
6. **Step 4 (Optional):** Add tags and description if needed
7. Click **Create Evaluation**

This Rule Prompt has no awareness of sarcasm, internet slang, or severity levels. That is intentional. You need a baseline that exposes the gaps human annotators will correct.

See [Custom Eval Metrics](https://docs.futureagi.com/docs/cookbook/quickstart/custom-eval-metrics) for more on creating custom evals.

Now prepare a test dataset of posts that will challenge the eval. Include sarcasm, slang, genuine threats, and a few unambiguous cases as controls.

Save this as `content_posts.csv`:

```csv
post_id,post_content,expected_label
1,"Just adopted the cutest golden retriever puppy! Meet Biscuit 🐕","safe"
2,"lol i'm literally dead 💀 this meme has me on the floor","safe"
3,"my mom's cooking could be classified as a war crime fr fr 😭","safe"
4,"that referee needs to be fired into the sun, absolute garbage calls all game","needs-review"
5,"she ate and left no crumbs 💅 queen behavior honestly","safe"
6,"if you support [political party] you're actually braindead, no cap","needs-review"
7,"I will find you and I will end your career. Sleep with one eye open.","violating"
8,"ratio + L + you fell off + nobody asked 🤡","needs-review"
```

Upload the dataset:

In [ ]:
import os
from fi.datasets import Dataset, DatasetConfig
from fi.utils.types import ModelTypes

dataset = Dataset(
    dataset_config=DatasetConfig(
        name="content-moderation-posts",
        model_type=ModelTypes.GENERATIVE_LLM,
    ),
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

dataset.create(source="content_posts.csv")

print(f"Dataset created: {dataset.dataset_config.name}")
print(f"Dataset ID: {dataset.dataset_config.id}")

## Step 2: Score every post with the baseline eval

Run your `content_appropriateness` eval across all posts. This is the "before" snapshot that you will compare against after incorporating human corrections.

In [ ]:
dataset.add_evaluation(
    name="appropriateness-v1",
    eval_template="content_appropriateness",
    required_keys_to_column_names={
        "post_content": "post_content",
    },
    model="turing_small",
    run=True,
    reason_column=True,
)

print("Evaluation 'appropriateness-v1' started. Check the dashboard for results.")

Once the evaluation completes, open the dataset in the dashboard to review. Each row now has a Pass/Fail score and a reason. With the naive Rule Prompt, you will likely see results like this:

| Post | Expected | AI Verdict | Issue |
|------|----------|-----------|-------|
| Post 1 (puppy adoption) | safe | Pass | Correct |
| Post 2 ("literally dead") | safe | Fail | Flags "dead" as violent language |
| Post 3 ("war crime") | safe | Fail | Flags "war crime" as violent content |
| Post 4 (referee into the sun) | needs-review | Fail | Reasonable flag, but too aggressive |
| Post 5 ("ate and left no crumbs") | safe | Pass or Fail | May misinterpret slang |
| Post 6 ("braindead") | needs-review | Fail | Correct to flag |
| Post 7 (explicit threat) | violating | Fail | Correct |
| Post 8 ("ratio + L") | needs-review | Fail | Flags internet slang as harassment |

The pattern is clear: the eval treats sarcasm, hyperbole, and internet slang the same way it treats genuine threats. Posts 2, 3, 5, and 8 are the problem cases.

Download the scored results for later comparison:

In [ ]:
df_v1 = dataset.download(load_to_pandas=True)
print("Columns:", list(df_v1.columns))
print(df_v1[["post_id", "post_content", "appropriateness-v1"]].to_string())

## Step 3: Have humans annotate the eval's mistakes

Now bring humans into the loop. Create an annotation workflow where human reviewers mark where the eval got it wrong and, critically, explain why.

Open the dataset and set up an annotation view:

1. Go to **Dataset** → click `content-moderation-posts`
2. Click the **Annotations** tab
3. Click **Create New View**
4. Name the view: "Content Moderation Review"

**Configure the view:**

**Static Fields**: select `post_id` and `expected_label` (visible to annotators, not editable).

**Response Fields**: select `post_content` (the content being evaluated).

**Labels**: click **New Label** for each:

| Label name | Type | Description |
|---|---|---|
| Human Verdict | Categorical | Categories: "Agree with AI", "Disagree - Actually Safe", "Disagree - Actually Violating", "Ambiguous" |
| Disagreement Reason | Text | If you disagree, explain what context the AI is missing |
| Confidence | Numeric (1-5) | 1 = very unsure, 5 = certain |

**Annotators**: add your team members. Each annotator labels rows independently.

Click **Save** to create the view.

See [Dataset Annotation](https://docs.futureagi.com/docs/cookbook/quickstart/dataset-annotation) for the full annotation setup.

Here is what the annotations look like for four key disagreements:

**Post 2: "lol i'm literally dead, this meme has me on the floor"**
- **Human Verdict**: "Disagree - Actually Safe"
- **Disagreement Reason**: "Standard Gen-Z hyperbole. 'Literally dead' and 'on the floor' mean finding something very funny. No actual violence."
- **Confidence**: 5

**Post 3: "my mom's cooking could be classified as a war crime fr fr"**
- **Human Verdict**: "Disagree - Actually Safe"
- **Disagreement Reason**: "Sarcastic joke about bad cooking. 'War crime' is used hyperbolically. 'fr fr' means 'for real.' Normal family humor."
- **Confidence**: 5

**Post 5: "she ate and left no crumbs, queen behavior honestly"**
- **Human Verdict**: "Disagree - Actually Safe"
- **Disagreement Reason**: "'Ate and left no crumbs' is slang for 'performed exceptionally well.' This is a compliment."
- **Confidence**: 5

**Post 8: "ratio + L + you fell off + nobody asked"**
- **Human Verdict**: "Ambiguous"
- **Disagreement Reason**: "Standard internet discourse. 'Ratio', 'L', and 'fell off' are competitive social media language. Dismissive but not targeted harassment. Context-dependent."
- **Confidence**: 3

Each annotation captures not just whether the AI was right or wrong, but *why*. That reasoning is the raw material for refining the eval.

**Tip:** Enable **Auto-Annotation** on the Human Verdict label. After your annotators label the first few rows, the platform learns the pattern and suggests labels for remaining rows. You can accept or override each suggestion.

## Step 4: Identify systematic patterns in the corrections

Export the annotated dataset and look for recurring themes in the disagreement reasons. The goal is to turn individual corrections into general rules.

In [ ]:
import os
import pandas as pd
from fi.datasets import Dataset

annotated = Dataset.get_dataset_config(
    "content-moderation-posts",
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

df = annotated.download(load_to_pandas=True)
print("Columns:", list(df.columns))
print(df.head())

Filter to the rows where humans disagreed:

In [ ]:
# Filter to rows where humans disagreed with the AI
disagree_cols = [c for c in df.columns if "human_verdict" in c.lower() or "Human Verdict" in c]

if disagree_cols:
    col = disagree_cols[0]
    disagreements = df[df[col].str.contains("Disagree", na=False)]
    print(f"Total disagreements: {len(disagreements)}")
    print(f"Total rows: {len(df)}")
    print(f"Disagreement rate: {len(disagreements)/len(df)*100:.0f}%")

From the annotations, three patterns emerge:

**Pattern 1: Sarcasm and hyperbole flagged as literal threats.** Posts 2 and 3 use words like "dead" and "war crime" in clearly non-literal ways. The eval has no instruction to distinguish figurative from literal language.

**Pattern 2: Internet slang misclassified as harmful.** Posts 5 and 8 use platform-specific slang ("ate and left no crumbs", "ratio + L") that the eval does not recognize. It defaults to flagging unfamiliar informal language.

**Pattern 3: No severity gradient.** The eval treats a sarcastic cooking joke and an explicit death threat with the same FAIL verdict. There is no instruction to weigh severity or consider intent.

These three patterns map directly to gaps in the Rule Prompt.

## Step 5: Revise the Rule Prompt to address every pattern annotators flagged

Replace the Rule Prompt with a version that addresses every pattern the annotators identified. Open the eval and paste in the refined prompt:

1. Go to **Evals** → click `content_appropriateness`
2. Edit the **Rule Prompt** and replace it with:

```
You are a content moderator for a social media platform used primarily by a young adult audience (18-30).

Evaluate this user post: {{post_content}}

IMPORTANT CONTEXT FOR EVALUATION:

1. SARCASM AND HYPERBOLE: Internet users frequently use exaggerated language for humor. Phrases like "I'm literally dead", "this killed me", "war crime" (about food/fashion/sports), "I'm going to scream", or "fire" are standard hyperbolic expressions, NOT literal threats or references to violence. If the surrounding context is clearly humorous or casual, treat exaggerated language as safe.

2. INTERNET AND GEN-Z SLANG: The following are common slang expressions that are NOT harmful:
   - "ate / ate and left no crumbs" = performed exceptionally well
   - "slay / queen / king" = compliments
   - "ratio / L / W" = competitive social media language (agreement/disagreement metrics)
   - "fell off" = declined in quality or relevance
   - "no cap / fr fr" = "for real" (emphasis)
   - "bruh / bestie / sis" = casual address
   - "it's giving" = it resembles or evokes
   These expressions should not be flagged unless combined with genuinely harmful content.

3. SEVERITY AND INTENT: Distinguish between:
   - Casual negativity or competitive banter (safe)
   - Directed insults that dehumanize or use slurs (needs review)
   - Explicit threats of physical harm with specific targets (violating)

Mark PASS if the post is appropriate for a general audience, even if it uses informal language, sarcasm, hyperbole, or internet slang.

Mark FAIL only if the post contains:
- Hate speech or slurs targeting a protected group
- Credible, specific threats of physical violence (not hyperbolic expressions)
- Graphic sexual content
- Sustained, targeted harassment of a specific individual (not general competitive banter)

When in doubt about sarcasm or slang, lean toward PASS. False negatives (missing a genuinely harmful post) are corrected in human review. False positives (flagging safe posts) erode user trust at scale.

Return a clear PASS/FAIL decision with a short reason.
```

3. Click **Update Evaluation**

Each section of the refined prompt addresses a specific pattern from the annotations:
- **Pattern 1** (sarcasm): Section 1 instructs the eval to recognize hyperbolic language
- **Pattern 2** (slang): Section 2 provides a glossary of common internet slang
- **Pattern 3** (severity): Section 3 introduces a three-tier severity framework

## Step 6: Re-evaluate and measure the improvement

Run the updated eval on the exact same dataset. Same posts, same expected labels. The only change is the Rule Prompt.

In [ ]:
import os
from fi.datasets import Dataset

dataset = Dataset.get_dataset_config(
    "content-moderation-posts",
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

dataset.add_evaluation(
    name="appropriateness-v2",
    eval_template="content_appropriateness",
    required_keys_to_column_names={
        "post_content": "post_content",
    },
    model="turing_small",
    run=True,
    reason_column=True,
)

print("Evaluation 'appropriateness-v2' started. Check the dashboard for results.")

Once complete, download and compare both versions side by side:

In [ ]:
df = dataset.download(load_to_pandas=True)

# Find the v1 and v2 eval columns
v1_col = [c for c in df.columns if "appropriateness-v1" in c and "reason" not in c.lower()]
v2_col = [c for c in df.columns if "appropriateness-v2" in c and "reason" not in c.lower()]

if v1_col and v2_col:
    comparison = df[["post_id", "post_content", "expected_label", v1_col[0], v2_col[0]]]
    print(comparison.to_string())

With the refined Rule Prompt, you should see clear improvement on the problem cases:

| Post | Expected | v1 | v2 | Fixed? |
|------|----------|-----|-----|--------|
| Post 1 (puppy) | safe | Pass | Pass | Already correct |
| Post 2 ("literally dead") | safe | Fail | Pass | Recognizes hyperbole |
| Post 3 ("war crime" cooking) | safe | Fail | Pass | Recognizes sarcasm |
| Post 4 (referee) | needs-review | Fail | Pass or Fail | Depends on severity read |
| Post 5 ("ate no crumbs") | safe | Fail | Pass | Recognizes slang |
| Post 6 ("braindead") | needs-review | Fail | Fail | Correct flag maintained |
| Post 7 (explicit threat) | violating | Fail | Fail | Correct flag maintained |
| Post 8 ("ratio + L") | needs-review | Fail | Pass | Recognizes banter |

You can also spot-check individual posts through the SDK:

In [ ]:
import os
from fi.evals import Evaluator

evaluator = Evaluator(
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

test_posts = [
    "lol i'm literally dead \ud83d\udc80 this meme has me on the floor",
    "she ate and left no crumbs \ud83d\udc85 queen behavior honestly",
    "I will find you and I will end your career. Sleep with one eye open.",
]

for post in test_posts:
    result = evaluator.evaluate(
        eval_templates="content_appropriateness",
        inputs={"post_content": post},
    )

    eval_result = result.eval_results[0]
    verdict = eval_result.output
    print(f"Post: {post[:60]}...")
    print(f"  Verdict: {verdict}")
    print(f"  Reason: {eval_result.reason}\n")

The sarcasm and slang posts should now pass, while the genuine threat still fails. One feedback cycle turned a noisy eval into a useful one.

## What you solved

You closed a full feedback loop: eval, annotate, refine, re-evaluate. Your custom eval went from flagging sarcasm as toxic to understanding the difference between jokes and genuine threats.

The loop you can repeat whenever the eval drifts:

```
Run eval → Humans correct mistakes → Identify patterns →
Update Rule Prompt → Re-evaluate to confirm improvement
```

Each cycle makes the eval more aligned with human judgment. The patterns your annotators identify (sarcasm, slang, cultural context, severity) become explicit instructions in the Rule Prompt.

## Explore further

- [Custom Eval Metrics](https://docs.futureagi.com/docs/cookbook/quickstart/custom-eval-metrics): Create domain-specific evals
- [Dataset Annotation](https://docs.futureagi.com/docs/cookbook/quickstart/dataset-annotation): Full annotation workflow
- [Batch Evaluation](https://docs.futureagi.com/docs/cookbook/quickstart/batch-eval): Evaluate datasets at scale
- [Running Your First Eval](https://docs.futureagi.com/docs/cookbook/quickstart/first-eval): Three evaluation engines